# 88 — Phái sinh và intraday với backtrader

`VN30F1M` khác cổ phiếu ở gần như mọi chỗ backtrader quan tâm:

1. **Giá là điểm chỉ số, khối lượng là hợp đồng.** Một điểm đáng 100.000 VND —
   con số đó không nằm trong dữ liệu, nó nằm ở `PhiPhaiSinhVN`
2. **Ký quỹ thay cho thanh toán đủ**, lãi/lỗ trả về tài khoản mỗi ngày, và
   **bán khống là chuyện bình thường** — chiến lược ở đây đi cả hai chiều
3. **Dữ liệu 1 phút** chỉ lấy được theo cửa sổ ≤ `max_intraday_days`, cột `time`
   mang múi giờ, và backtrader phải tự gộp 1 phút thành 15 phút / một ngày
4. **Chuỗi `VN30F1M` là chuỗi nối** — mỗi tháng nó nhảy từ hợp đồng sắp đáo
   hạn sang hợp đồng kế tiếp

Ba cạm bẫy được chứng minh bằng số: **(a)** bỏ múi giờ sai cách làm thanh lệch
7 tiếng và luật "đóng vị thế trước ATC" không bao giờ chạy · **(b)** bước nhảy
của chuỗi nối ở ngày đáo hạn sinh tín hiệu giả · **(c)** gọi trong giờ giao dịch
nhận về một thanh của phiên chưa đóng.

Notebook import mọi thứ dùng chung từ `finlens_examples/bt_vn.py` (gom ở các
notebook [`81`](81_tu_finlens_toi_cerebro.ipynb)–[`83`](83_luat_choi_viet_nam.ipynb)).
Mục 3 đo một lỗi định giá tài khoản mà bản đầu của `PhiPhaiSinhVN` mắc phải
(module đã sửa sau khi notebook này phát hiện) — và vì sao sửa một nửa thì vô ích.

In [1]:
import datetime as dt
import sys
import time
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import backtrader as bt
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import finlens
from finlens_examples import ap_dung_theme, hom_nay, lui_ngay, nen, thanh_doi_mau
from finlens_examples import bt_vn
from finlens_examples.charts import CHUOI

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

HD = "VN30F1M"
print(f"Hôm nay theo server: {HOM_NAY} · backtrader {bt.__version__} · finlens {finlens.__version__}")

D:\finlens\finlens-python\finlens-python-example\venv\Lib\site-packages\anyio\_core\_tasks.py:275: UpgradeAvailableWarning: finlens 1.5.0 đã có (đang dùng 1.4.0). Nâng cấp: pip install -U finlens
  retval = await self._coro


Hôm nay theo server: 2026-10-20 · backtrader 1.9.78.123 · finlens 1.4.0


## 1 · Thông số thể chế — cái nào là đặc tả, cái nào là giả định

Không thông số nào dưới đây có trong dữ liệu finlens. Mỗi dòng ghi rõ nguồn;
dòng nào không xác minh được thì là **tham số**, đổi một chỗ là cả notebook
chạy lại theo.

| Thông số | Giá trị dùng | Nguồn / trạng thái |
|---|---|---|
| Hệ số nhân | **100.000 VND/điểm** | Đặc tả HĐTL chỉ số VN30 của HNX ([bản tóm tắt của MBS, 2025](https://mbs.com.vn/files/uploads/2025/05/1.-Quy-dinh-giao-dich-hdtl-csvn30.pdf)) |
| Giờ giao dịch | ATO 8:45–9:00 · liên tục 9:00–11:30 và 13:00–14:30 · **ATC 14:30–14:45** | Như trên; mục 4 kiểm lại bằng giờ của chính các thanh 1 phút |
| Ngày giao dịch cuối cùng | **thứ Năm thứ ba** của tháng đáo hạn, trùng ngày nghỉ thì lùi về phiên liền trước | Như trên; mục 10 kiểm bằng dữ liệu |
| Giá thanh toán cuối cùng | trung bình VN30 trong 30 phút cuối (15 phút liên tục + 15 phút ATC), bỏ 3 giá trị cao nhất và 3 thấp nhất | Như trên — vì thế basis ngày đáo hạn **không** về đúng 0 |
| Tỷ lệ ký quỹ | **20%** — *giả định* | VSDC công bố "theo từng thời kỳ". Mức chính thức gần nhất tìm được là **17%**, áp dụng từ 15/12/2022 ([nhadautu.vn](https://nhadautu.vn/nang-ty-le-ky-quy-chung-khoan-phai-sinh-tu-13-len-17-d72538.html)); công ty chứng khoán thường thu cao hơn mức VSDC. Không tìm được văn bản 2026 |
| Phí mỗi hợp đồng mỗi chiều | **10.000 VND** — *giả định* | Mặc định của `bt_vn.PhiPhaiSinhVN`, gộp môi giới + Sở + VSDC. Kiểm biểu phí công ty chứng khoán của bạn |

In [2]:
HE_SO_NHAN = 100_000                  # VND mỗi điểm — đặc tả sản phẩm
TY_LE_KY_QUY = 0.20                   # GIẢ ĐỊNH — VSDC công bố theo thời kỳ (gần nhất tìm được: 17%, 12/2022)
PHI_HD = 10_000                       # GIẢ ĐỊNH — VND/hợp đồng/chiều

GIO_ATC = dt.time(14, 30)             # hết khớp liên tục; 14:30–14:45 là khớp định kỳ đóng cửa
GIO_DONG_VI_THE = dt.time(14, 25)     # chiến lược tự đóng từ đây — lệnh thị trường khớp ở thanh kế tiếp
GIO_CUOI_VAO_LENH = dt.time(14, 0)    # không mở vị thế mới sau giờ này
VON = 100_000_000                     # VND

print(f"Hệ số nhân {HE_SO_NHAN:,} VND/điểm · ký quỹ {TY_LE_KY_QUY:.0%} · phí {PHI_HD:,} VND/HĐ/chiều · "
      f"đóng vị thế từ {GIO_DONG_VI_THE:%H:%M}, ATC từ {GIO_ATC:%H:%M} · vốn {VON:,} VND")

Hệ số nhân 100,000 VND/điểm · ký quỹ 20% · phí 10,000 VND/HĐ/chiều · đóng vị thế từ 14:25, ATC từ 14:30 · vốn 100,000,000 VND


## 2 · Dữ liệu EOD, và ⚠️ cạm bẫy (c): thanh của phiên chưa đóng

`client.eod.derivative.ohlcv` trả giá theo `index_point`, khối lượng theo
`contract`. `bt_vn.chuan_bi_frame` đọc đơn vị này từ `df.attrs` và **giữ nguyên**
giá (chỉ `kVND` mới bị nhân 1.000).

Gọi trong giờ giao dịch, dòng cuối là **phiên đang chạy**: giá "đóng cửa" của
nó là giá khớp gần nhất, khối lượng mới là một phần. Backtest trên thanh đó là
backtest trên một giá đóng cửa chưa tồn tại.

In [3]:
eod_tho = client.eod.derivative.ohlcv(HD, start=lui_ngay(HOM_NAY, nam=6))
meta = eod_tho.attrs["finlens"]
print("Đơn vị:", {k: v for k, v in meta["units"].items() if v})
print(f"price_basis={meta['price_basis']} · as_of={meta['as_of']} · warnings={meta['warnings']}")

bay_gio = dt.datetime.fromisoformat(client.whoami()["server_time"])
cuoi = eod_tho.iloc[-1]
kl_truoc = eod_tho["volume"].iloc[-21:-1]
print(f"\nGiờ server: {bay_gio:%Y-%m-%d %H:%M}")
print(f"Thanh cuối {cuoi['date']:%Y-%m-%d}: close={cuoi['close']:.1f} · khối lượng {cuoi['volume']:,.0f} HĐ")
print(f"Phiên liền trước: {eod_tho['volume'].iloc[-2]:,.0f} HĐ · trung vị 20 phiên trước {kl_truoc.median():,.0f} HĐ "
      f"→ thanh cuối bằng {cuoi['volume'] / kl_truoc.median():.1%}")

eod = bt_vn.bo_phien_chua_dong(eod_tho, client)
print(f"\nbo_phien_chua_dong: {len(eod_tho)} → {len(eod)} thanh (bỏ {len(eod_tho) - len(eod)}) · "
      f"phiên cuối còn lại {eod['date'].max():%Y-%m-%d}")

Đơn vị: {'open': 'index_point', 'high': 'index_point', 'low': 'index_point', 'close': 'index_point', 'volume': 'contract'}
price_basis=raw · as_of=2026-10-20T00:00:00+07:00 · warnings=[]



Giờ server: 2026-10-20 11:22
Thanh cuối 2026-10-20: close=1837.7 · khối lượng 103,148 HĐ
Phiên liền trước: 175,662 HĐ · trung vị 20 phiên trước 213,942 HĐ → thanh cuối bằng 48.2%



bo_phien_chua_dong: 1497 → 1496 thanh (bỏ 1) · phiên cuối còn lại 2026-10-19


Hai điều đáng để ý trong output trên:

- Nếu notebook chạy trước 15:00, thanh cuối có khối lượng thấp hơn phiên bình
  thường — càng sớm trong phiên càng thấp, output in đúng tỷ lệ lúc chạy — và đó
  là dấu hiệu duy nhất cho thấy nó chưa xong. Chạy sau 15:00 thì
  `bo_phien_chua_dong` không bỏ gì, đúng như mong muốn.
- **EOD không cảnh báo gì** (`warnings=[]`). Intraday thì có — mục 4 sẽ thấy cảnh
  báo `SESSION_IN_PROGRESS` ngay trong `df.attrs`. Đừng trông vào cảnh báo ở EOD;
  quyết định "phiên đã đóng chưa" bằng đồng hồ server.

"Dở dang" nghĩa là gì cụ thể: thanh hôm nay của EOD không phải một ảnh chụp lúc
nào đó trong phiên, nó **chạy theo khớp lệnh**. Gọi hai lần cách nhau vài giây
(`refresh=True` để bỏ qua cache của thư viện), rồi đặt cạnh intraday cùng lúc:

In [4]:
lan = []
for i in range(2):
    luc = client.whoami()["server_time"][11:19]
    e = client.eod.derivative.ohlcv(HD, start=HOM_NAY.isoformat(), refresh=True)
    lan.append(dict(luc=luc, ngay=f"{e['date'].iloc[-1]:%Y-%m-%d}", close=e["close"].iloc[-1],
                    volume=e["volume"].iloc[-1], as_of=e.attrs["finlens"]["as_of"],
                    warnings=len(e.attrs["finlens"]["warnings"])))
    if i == 0:
        time.sleep(5)
print(pd.DataFrame(lan).set_index("luc").to_string())
print(f"Khối lượng EOD tăng {lan[1]['volume'] - lan[0]['volume']:,.0f} HĐ giữa hai lần gọi")

it = client.intraday.derivative.ohlcv(HD, start=HOM_NAY.isoformat(), end=HOM_NAY.isoformat(), refresh=True)
if len(it):
    print(f"Intraday cùng lúc: {len(it)} thanh tới {it['time'].max():%H:%M}, close={it['close'].iloc[-1]}, "
          f"tổng khối lượng {it['volume'].sum():,.0f} HĐ, as_of={it.attrs['finlens']['as_of']}")

                ngay   close    volume                      as_of  warnings
luc                                                                        
11:22:51  2026-10-20  1837.8  103158.0  2026-10-20T00:00:00+07:00         0
11:22:57  2026-10-20  1837.7  103188.0  2026-10-20T00:00:00+07:00         0
Khối lượng EOD tăng 30 HĐ giữa hai lần gọi
Intraday cùng lúc: 140 thanh tới 11:19, close=1838.7, tổng khối lượng 102,623 HĐ, as_of=2026-10-20T11:19:00+07:00


C:\Program Files\Python312\Lib\asyncio\events.py:88: DataQualityWarning: [SESSION_IN_PROGRESS] Phiên ngày 2026-10-20 đang chạy (giờ phiên 09:00–15:00), nên khoảng bạn yêu cầu còn thay đổi: gọi lại đúng khoảng này sau 15:00 sẽ ra số khác. `meta.as_of` đang là 2026-10-20T11:19:00+07:00. Đừng cache kết quả này như thể nó đã chốt.
  self._context.run(self._callback, *self._args)


Trong giờ giao dịch, khối lượng của thanh EOD tăng ngay giữa hai lần gọi cách
nhau vài giây, và EOD còn **mới hơn** intraday (intraday chỉ có tới thanh 1 phút
đã xong gần nhất, `as_of` theo phút). Nhưng metadata của EOD không phản ánh gì:
`as_of` đứng ở nửa đêm, `warnings` rỗng. Chạy ngoài giờ giao dịch thì hai lần gọi
giống hệt.

Thanh dở dang hỏng tín hiệu thế nào? Tín hiệu EOD ở mục 11 dựa trên thay đổi giá
đóng cửa so với độ lệch chuẩn 20 phiên. Tính nó trên cả hai frame:

In [5]:
def z_thay_doi(close: pd.Series, cua_so: int = 20) -> pd.Series:
    # Thay đổi điểm hôm nay chia độ lệch chuẩn (tổng thể, như bt.ind.StdDev) của cửa_so phiên TRƯỚC
    delta = close.diff()
    return delta / delta.rolling(cua_so).std(ddof=0).shift(1)

z_tho = z_thay_doi(eod_tho["close"]).iloc[-1]
z_dung = z_thay_doi(eod["close"]).iloc[-1]
print(f"z của thanh cuối, frame thô       : {z_tho:+.2f}  ({eod_tho['date'].iloc[-1]:%Y-%m-%d})")
print(f"z của thanh cuối, đã bỏ phiên dở  : {z_dung:+.2f}  ({eod['date'].iloc[-1]:%Y-%m-%d})")
print("→ con số thứ nhất đổi mỗi lần chạy lại trong phiên; con số thứ hai thì không.")

z của thanh cuối, frame thô       : -0.32  (2026-10-20)
z của thanh cuối, đã bỏ phiên dở  : -1.31  (2026-10-19)
→ con số thứ nhất đổi mỗi lần chạy lại trong phiên; con số thứ hai thì không.


## 3 · `PhiPhaiSinhVN` — kiểm bằng tay, và một lỗi định giá tài khoản

`bt_vn.PhiPhaiSinhVN` đặt `stocklike=False`, nên broker của backtrader chạy chế độ
hợp đồng tương lai:

- mở vị thế **phong toả ký quỹ** `giá × 100.000 × tỷ_lệ` mỗi hợp đồng thay vì trả đủ
- cuối mỗi thanh **thanh toán lãi/lỗ theo giá thị trường** vào tiền mặt
  (`Δđiểm × 100.000 × số HĐ`)
- đóng vị thế trả lại ký quỹ; phí là `phi_hd` mỗi hợp đồng mỗi chiều

Kiểm trên năm phiên thật gần nhất: mua 1 hợp đồng ở thanh đầu (khớp giá mở cửa
thanh 2), đóng ở thanh 4 (khớp giá mở cửa thanh 5) — chạy ba lần, ba cách gắn phí:

| Cách gắn | Là gì |
|---|---|
| `PhiPhaiSinhCu` + `addcommissioninfo` | tái hiện **bản cũ** của `bt_vn`: `getvalue` mặc định của backtrader, `shortcash` mặc định |
| `PhiPhaiSinhVN` + `addcommissioninfo` | lớp hiện tại nhưng gắn trần — `shortcash` vẫn mặc định |
| `bt_vn.gan_phi_phai_sinh(cerebro, ...)` | cách đúng: lớp hiện tại + `shortcash=False` |

In [6]:
KW_PHI = dict(mult=HE_SO_NHAN, ty_le_ky_quy=TY_LE_KY_QUY, phi_hd=PHI_HD)


class PhiPhaiSinhCu(bt_vn.PhiPhaiSinhVN):
    # Tái hiện bt_vn.PhiPhaiSinhVN bản cũ: định giá vị thế bằng getvalue mặc định của backtrader
    getvalue = bt.CommInfoBase.getvalue


def gan_phi(ce: bt.Cerebro, cach: str = "dung") -> None:
    if cach == "cu":
        ce.broker.addcommissioninfo(PhiPhaiSinhCu(**KW_PHI))
    elif cach == "tran":
        ce.broker.addcommissioninfo(bt_vn.PhiPhaiSinhVN(**KW_PHI))
    else:
        bt_vn.gan_phi_phai_sinh(ce, **KW_PHI)


class MuaGiuBa(bt.Strategy):
    def __init__(self):
        self.dong, self.khop, self.vong = [], [], []

    def notify_order(self, o):
        if o.status == o.Completed:
            self.khop.append(dict(size=o.executed.size, gia=o.executed.price,
                                  phi=o.executed.comm, ky_quy=o.executed.value))

    def notify_trade(self, t):
        if t.isclosed:
            self.vong.append(dict(pnl=t.pnl, pnlcomm=t.pnlcomm))

    def next(self):
        if len(self) == 1:
            self.buy(size=1)
        elif len(self) == 4:
            self.close()
        tien = self.broker.getcash()
        ky_quy = sum(k["ky_quy"] * np.sign(k["size"]) for k in self.khop)  # đang phong toả
        self.dong.append(dict(ngay=self.data.datetime.date(0), close=self.data.close[0], vi_the=self.position.size,
                              tien_mat=tien, getvalue=self.broker.getvalue(), von_that=tien + ky_quy))


def chay_mau(cach):
    ce = bt.Cerebro(stdstats=False)
    ce.adddata(bt_vn.feed_tu_finlens(eod.tail(5)))
    ce.broker.setcash(VON)
    gan_phi(ce, cach)
    ce.addstrategy(MuaGiuBa)
    return ce.run()[0]

s_goc = chay_mau("cu")
s_tran = chay_mau("tran")
s_dung = chay_mau("dung")

bang = pd.DataFrame(s_goc.dong).set_index("ngay")
bang["getvalue_tran"] = [d["getvalue"] for d in s_tran.dong]
bang["getvalue_dung"] = [d["getvalue"] for d in s_dung.dong]
print((bang / [1, 1, 1e6, 1e6, 1e6, 1e6, 1e6]).round(3).rename(
    columns={"tien_mat": "tiền mặt (tr)", "getvalue": "getvalue cũ (tr)", "von_that": "vốn thật (tr)",
             "getvalue_tran": "getvalue gắn trần (tr)", "getvalue_dung": "getvalue gan_phi_phai_sinh (tr)"}).to_string())

             close  vi_the  tiền mặt (tr)  getvalue cũ (tr)  vốn thật (tr)  getvalue gắn trần (tr)  getvalue gan_phi_phai_sinh (tr)
ngay                                                                                                                               
2026-10-13  1867.5     0.0         100.00           100.000         100.00                 100.000                           100.00
2026-10-14  1857.0     1.0          61.65            98.790          98.99                  98.790                            98.99
2026-10-15  1839.8     1.0          59.93            96.726          97.27                  96.726                            97.27
2026-10-16  1855.1     1.0          61.46            98.562          98.80                  98.562                            98.80
2026-10-19  1841.0     0.0          99.74            99.740          99.74                  99.740                            99.74


In [7]:
vao, ra = s_goc.khop
mo = eod.tail(5)["open"].to_numpy()
ky_quy_tay = vao["gia"] * HE_SO_NHAN * TY_LE_KY_QUY
lai_tay = (ra["gia"] - vao["gia"]) * HE_SO_NHAN * vao["size"] - 2 * PHI_HD

print(f"Khớp mua  {vao['gia']:.1f} (= open thanh 2: {mo[1]:.1f}) · khớp bán {ra['gia']:.1f} (= open thanh 5: {mo[4]:.1f})")
print(f"Ký quỹ phong toả  — backtrader {vao['ky_quy']:,.0f} · tay {ky_quy_tay:,.0f} VND")
print(f"Lãi/lỗ sau phí    — backtrader {s_goc.vong[0]['pnlcomm']:,.0f} · tay "
      f"({ra['gia']:.1f} − {vao['gia']:.1f}) × 100.000 − 2 × {PHI_HD:,} = {lai_tay:,.0f} VND")
print(f"Tiền mặt cuối     — {s_goc.broker.getcash():,.0f} = vốn + lãi/lỗ: {VON + lai_tay:,.0f}")

print()
for ten, cot in [("cũ", "getvalue"), ("gắn trần", "getvalue_tran"), ("gan_phi_phai_sinh", "getvalue_dung")]:
    print(f"getvalue() {ten:<18}: lệch vốn thật tối đa "
          f"{(bang[cot] - bang['von_that']).abs().max():,.0f} VND khi còn vị thế")
lech = (bang["getvalue"] - bang["von_that"]).abs()
ngay_x = lech.idxmax()
tam_that = bang.loc[ngay_x, "von_that"] - (VON - vao["phi"])   # lãi/lỗ chưa chốt thật
tam_goc = bang.loc[ngay_x, "getvalue"] - (VON - vao["phi"])    # lãi/lỗ chưa chốt theo getvalue cũ
print(f"Ngày {ngay_x}: lãi/lỗ chưa chốt thật {tam_that:,.0f} VND · getvalue cũ thể hiện {tam_goc:,.0f} VND "
      f"(× {tam_goc / tam_that:.2f}, vị thế long)")

Khớp mua  1867.0 (= open thanh 2: 1867.0) · khớp bán 1864.6 (= open thanh 5: 1864.6)
Ký quỹ phong toả  — backtrader 37,340,000 · tay 37,340,000 VND
Lãi/lỗ sau phí    — backtrader -260,000 · tay (1864.6 − 1867.0) × 100.000 − 2 × 10,000 = -260,000 VND
Tiền mặt cuối     — 99,740,000 = vốn + lãi/lỗ: 99,740,000

getvalue() cũ                : lệch vốn thật tối đa 544,000 VND khi còn vị thế
getvalue() gắn trần          : lệch vốn thật tối đa 544,000 VND khi còn vị thế
getvalue() gan_phi_phai_sinh : lệch vốn thật tối đa 0 VND khi còn vị thế
Ngày 2026-10-15: lãi/lỗ chưa chốt thật -2,720,000 VND · getvalue cũ thể hiện -3,264,000 VND (× 1.20, vị thế long)


Lãi/lỗ, ký quỹ và phí **đúng tới từng đồng** ở cả ba cách. Chỗ khác nhau là
`broker.getvalue()` khi còn vị thế mở — và đó là thứ mọi analyzer đọc.

Mặc định, backtrader định giá vị thế tương lai bằng `abs(size) × get_margin(giá
đóng cửa hiện tại)`. Với ký quỹ cố định theo hợp đồng (cách backtrader được thiết
kế) thì không sao; với ký quỹ **theo tỷ lệ giá** thì "ký quỹ" được tính lại theo
giá mỗi thanh, và giá trị tài khoản lệch đúng `(giá hiện tại − giá vào) × 100.000
× 20% × số HĐ`: vị thế **long** thấy lãi/lỗ chưa chốt **phóng to 1,2 lần**, vị
thế **short** thấy nó **thu nhỏ còn 0,8 lần** (dòng cuối của output trên đo đúng
tỷ lệ đó). Bản cũ của `bt_vn.PhiPhaiSinhVN` mắc đúng lỗi này.

Hệ quả: `TimeReturn`, `DrawDown`, `SharpeRatio` sai trong lúc giữ vị thế, còn tiền
mặt cuối cùng thì vẫn đúng. Chiến lược intraday đóng hết mỗi tối nên lợi suất
ngày không bị ảnh hưởng; chiến lược giữ qua đêm ở mục 11 thì có — ở đó đo lại.

`bt_vn` hiện sửa bằng **hai phần**, và cột "gắn trần" cho thấy thiếu phần thứ hai
thì phần thứ nhất vô dụng:

1. `PhiPhaiSinhVN.getvalue()` định giá bằng ký quỹ theo giá vào lệnh
   `position.price` — đúng bằng số tiền đang bị phong toả;
2. `broker.set_shortcash(False)` — với mặc định `shortcash=True`, broker không gọi
   `getvalue()` mà gọi `getvaluesize(size, giá)`, một hàm không nhận vị thế nên
   không thể biết giá vào lệnh. Với hợp đồng tương lai, `shortcash` không đổi gì
   khác: phong toả và hoàn ký quỹ vẫn tính bằng cùng một công thức.

`bt_vn.gan_phi_phai_sinh(cerebro, ...)` làm cả hai. Từ đây notebook gắn phí bằng nó.

## 4 · Dữ liệu 1 phút: cửa sổ ≤ `max_intraday_days`, tính cả hai đầu

`client.intraday.derivative.ohlcv` nhận `interval` từ `1min` tới `4h`, nhưng mỗi
lời gọi chỉ được ≤ `max_intraday_days` ngày **tính cả `start` lẫn `end`** (xem
[`41`](../04-intraday-phai-sinh-vi-mo/41_intraday_order_flow.ipynb)). Hai mốc cách nhau
đúng 10 ngày là cửa sổ 11 ngày:

In [8]:
SO_NGAY_TOI_DA = client.limits()["max_intraday_days"]
BUOC = SO_NGAY_TOI_DA - 1   # end = start + 9 ngày → cửa sổ 10 ngày
print(f"max_intraday_days = {SO_NGAY_TOI_DA} → bước {BUOC} ngày")

a = HOM_NAY - dt.timedelta(days=30)
try:
    client.intraday.derivative.ohlcv(HD, start=a.isoformat(), end=(a + dt.timedelta(days=SO_NGAY_TOI_DA)).isoformat())
except finlens.InvalidDateRangeError as e:
    print(f"\nend = start + {SO_NGAY_TOI_DA} ngày → {type(e).__name__}: {str(e)[:110]}…")
ok = client.intraday.derivative.ohlcv(HD, start=a.isoformat(), end=(a + dt.timedelta(days=BUOC)).isoformat())
print(f"end = start + {BUOC} ngày  → {len(ok):,} thanh, truncated={ok.attrs['finlens']['truncated']}")

max_intraday_days = 10 → bước 9 ngày


end = start + 10 ngày → InvalidDateRangeError: [FL_VALIDATION_DATE_RANGE] Khoảng bạn yêu cầu dài 11 ngày, nhưng dữ liệu trong phiên chỉ lấy được tối đa 10 ng…


end = start + 9 ngày  → 1,687 thanh, truncated=False


In [9]:
def tai_intraday(start: dt.date, end: dt.date, interval: str = "1min") -> tuple[pd.DataFrame, pd.DataFrame]:
    # Lặp cửa sổ ≤ max_intraday_days; ghi lại số dòng, truncated và cảnh báo của từng cửa sổ
    phan, so = [], []
    a = start
    while a <= end:
        b = min(a + dt.timedelta(days=BUOC), end)
        f = client.intraday.derivative.ohlcv(HD, start=a.isoformat(), end=b.isoformat(), interval=interval)
        m = f.attrs["finlens"]
        so.append(dict(start=a, end=b, so_dong=len(f), truncated=m["truncated"],
                       canh_bao=",".join(w["code"] for w in m["warnings"])))
        phan.append(f)
        a = b + dt.timedelta(days=1)
    return pd.concat(phan, ignore_index=True), pd.DataFrame(so)

t0 = time.perf_counter()
BAT_DAU_ID = HOM_NAY - dt.timedelta(days=365)
m1_tho, so_cua_so = tai_intraday(BAT_DAU_ID, HOM_NAY)
print(f"{len(so_cua_so)} cửa sổ · {len(m1_tho):,} thanh 1 phút · {time.perf_counter() - t0:.0f} giây")
print(f"Cửa sổ truncated: {int(so_cua_so['truncated'].sum())} · dòng nhiều nhất một cửa sổ: "
      f"{so_cua_so['so_dong'].max():,} (max_rows = {client.limits()['max_rows']:,})")
print(f"Cảnh báo: {so_cua_so.loc[so_cua_so['canh_bao'] != '', ['start', 'end', 'canh_bao']].to_dict('records')}")
print(f"Trùng mốc thời gian giữa các cửa sổ: {int(m1_tho.duplicated(['time']).sum())}")

C:\Program Files\Python312\Lib\asyncio\events.py:88: DataQualityWarning: [SESSION_IN_PROGRESS] Phiên ngày 2026-10-20 đang chạy (giờ phiên 09:00–15:00), nên khoảng bạn yêu cầu còn thay đổi: gọi lại đúng khoảng này sau 15:00 sẽ ra số khác. `meta.as_of` đang là 2026-10-20T11:19:00+07:00. Đừng cache kết quả này như thể nó đã chốt.
  self._context.run(self._callback, *self._args)


37 cửa sổ · 58,669 thanh 1 phút · 24 giây


Cửa sổ truncated: 0 · dòng nhiều nhất một cửa sổ: 1,928 (max_rows = 100,000)
Cảnh báo: [{'start': datetime.date(2026, 10, 15), 'end': datetime.date(2026, 10, 20), 'canh_bao': 'SESSION_IN_PROGRESS'}]
Trùng mốc thời gian giữa các cửa sổ: 0


`pd.concat` vừa làm mất `df.attrs` (xem ghi chú "Hành vi chưa được tài liệu hoá"
trong `ROADMAP.md`). `bt_vn.chuan_bi_frame` từ chối đoán đơn vị — với cổ phiếu
đoán sai là lệch 1.000 lần:

In [10]:
print(f"attrs sau concat: {m1_tho.attrs}")
try:
    bt_vn.chuan_bi_frame(m1_tho)
except ValueError as e:
    print(f"ValueError: {e}")
DON_VI = ok.attrs["finlens"]["units"]["close"]  # đọc từ frame một cửa sổ, còn attrs
print(f"\n→ truyền don_vi_gia='{DON_VI}' (đọc từ frame một cửa sổ, không gõ tay)")

attrs sau concat: {}
ValueError: Frame không còn df.attrs['finlens'] (thường do pd.concat/merge). Truyền don_vi_gia='kVND' | 'VND' | 'index_point' cho rõ.

→ truyền don_vi_gia='index_point' (đọc từ frame một cửa sổ, không gõ tay)


### Lịch phiên: đối chiếu intraday với EOD

Trước khi backtest, hỏi dữ liệu xem nó có đủ phiên không. Lịch chuẩn là các
phiên có trong EOD (đã bỏ phiên đang chạy).

In [11]:
m1_tho["ngay"] = m1_tho["time"].dt.tz_localize(None).dt.normalize()
lich_eod = pd.DatetimeIndex(eod["date"])
ngay_id = pd.DatetimeIndex(m1_tho["ngay"].unique())
trong_ky = lich_eod[(lich_eod >= ngay_id.min()) & (lich_eod <= ngay_id.max())]

thua = ngay_id.difference(trong_ky).difference([pd.Timestamp(bay_gio.date())])
thieu = trong_ky.difference(ngay_id)
print(f"Có ở intraday, không có ở EOD : {[f'{d:%Y-%m-%d} ({d:%a})' for d in thua]}")
print(f"Có ở EOD, không có ở intraday : {[f'{d:%Y-%m-%d}' for d in thieu]}")

so_thanh = m1_tho[m1_tho["ngay"].isin(trong_ky)].groupby("ngay").size()
CHUAN = int(so_thanh.mode().iloc[0])
print(f"\nSố thanh một phiên đủ: {CHUAN} ({(so_thanh == CHUAN).sum()}/{len(so_thanh)} phiên)")
print("Phiên thiếu thanh:", {f"{d:%Y-%m-%d}": int(n) for d, n in so_thanh[so_thanh < CHUAN].items()})

Có ở intraday, không có ở EOD : ['2025-11-01 (Sat)']
Có ở EOD, không có ở intraday : ['2026-04-10', '2026-04-14', '2026-04-15']

Số thanh một phiên đủ: 241 (228/245 phiên)
Phiên thiếu thanh: {'2025-10-21': 240, '2026-02-25': 150, '2026-03-16': 238, '2026-03-18': 240, '2026-03-19': 231, '2026-03-30': 240, '2026-04-06': 237, '2026-04-09': 166, '2026-04-13': 42, '2026-05-13': 185, '2026-06-11': 239, '2026-06-17': 221, '2026-07-01': 187, '2026-07-31': 239, '2026-08-04': 237, '2026-10-02': 240, '2026-10-06': 240}


In [12]:
# Giờ phiên — kiểm bằng chính các thanh, không tin bảng ở mục 1
gio = m1_tho["time"].dt.strftime("%H:%M")
phut = m1_tho["time"].dt.hour * 60 + m1_tho["time"].dt.minute
print(f"Thanh sớm nhất {gio.min()} · muộn nhất {gio.max()}")
print(f"Thanh trong 11:30–12:59: {int(((phut >= 690) & (phut < 780)).sum())} · trong 14:30–14:44: "
      f"{int(((phut >= 870) & (phut < 885)).sum())} · đúng 14:45: {int((gio == '14:45').sum())}")
print(f"Thanh liên tục cuối buổi chiều: {gio[(phut >= 780) & (phut < 870)].max()}")

Thanh sớm nhất 09:00 · muộn nhất 14:45
Thanh trong 11:30–12:59: 0 · trong 14:30–14:44: 0 · đúng 14:45: 240
Thanh liên tục cuối buổi chiều: 14:29


Dữ liệu khớp với quy định: thanh 1 phút được **đánh nhãn theo phút bắt đầu**,
khớp liên tục chạy 09:00–11:29 và 13:00–14:29, rồi **một thanh duy nhất lúc 14:45**
là kết quả khớp định kỳ đóng cửa (ATC). Không có thanh nào 14:30–14:44 — trong
ATC không có giá khớp liên tục để mà đóng vị thế. Vì thế chiến lược phải đặt
lệnh đóng **từ 14:25** để khớp ở thanh kế tiếp, trước 14:30.

Có một "phiên" chỉ có ở intraday (nếu output ở trên liệt kê một ngày thứ Bảy, đó
là dữ liệu lạc) và vài phiên thiếu thanh. Phiên thiếu buổi chiều nghĩa là không
có thanh 14:25 nào để kích hoạt lệnh đóng — chiến lược ở mục 8 có thêm một chốt
chặn cho trường hợp đó. Giữ lại các phiên có trong lịch EOD, rồi bỏ phiên đang
chạy:

In [13]:
m1 = m1_tho[m1_tho["ngay"].isin(lich_eod) | (m1_tho["ngay"] == pd.Timestamp(bay_gio.date()))].drop(columns="ngay")
m1_xong = bt_vn.bo_phien_chua_dong(m1, client, cot="time")
print(f"{len(m1_tho):,} → lọc lịch EOD {len(m1):,} → bỏ phiên đang chạy {len(m1_xong):,} thanh "
      f"(bỏ {len(m1) - len(m1_xong)} thanh của hôm nay)")

khung_1p = bt_vn.chuan_bi_frame(m1_xong, don_vi_gia=DON_VI)
print(f"\nkhung_1p: {khung_1p.index.min()} → {khung_1p.index.max()} · index tz={khung_1p.index.tz} · "
      f"{khung_1p.index.normalize().nunique()} phiên")
khung_1p.head(3)

58,669 → lọc lịch EOD 58,660 → bỏ phiên đang chạy 58,520 thanh (bỏ 140 thanh của hôm nay)



khung_1p: 2025-10-20 09:00:00 → 2026-10-19 14:45:00 · index tz=None · 245 phiên


,open,high,low,close,volume
datetime,,,,,
2025-10-20 09:00:00,1969.1,1969.9,1961.1,1963.4,4823.0
2025-10-20 09:01:00,1963.0,1963.5,1960.0,1962.2,2824.0
2025-10-20 09:02:00,1962.2,1962.8,1959.3,1961.9,2083.0


## 5 · ⚠️ Cạm bẫy (a): múi giờ

Cột `time` có kiểu `datetime64[ns, Asia/Ho_Chi_Minh]`. backtrader làm việc với
datetime **naive**. Có ba cách quen tay để "bỏ múi giờ", và chỉ một cách giữ được
giờ Việt Nam:

In [14]:
t = m1_xong["time"].sort_values()
cach = {
    "bt_vn.chuan_bi_frame (tz_convert VN → tz_localize(None))": khung_1p.index,
    "tz_convert('UTC').tz_localize(None)": pd.DatetimeIndex(t.dt.tz_convert("UTC").dt.tz_localize(None)),
    ".values": pd.DatetimeIndex(t.values),
}

# Cách thứ tư: đưa thẳng index có múi giờ vào PandasData và để backtrader tự xử lý
class GhiGio(bt.Strategy):
    def __init__(self):
        self.gio = []
    def next(self):
        self.gio.append(self.data.datetime.datetime(0))

N0 = int((khung_1p.index.normalize() == khung_1p.index.normalize()[0]).sum())  # số thanh của phiên đầu
ce = bt.Cerebro(stdstats=False)
co_tz = khung_1p.iloc[:N0].tz_localize("Asia/Ho_Chi_Minh")
ce.adddata(bt.feeds.PandasData(dataname=co_tz, openinterest=None, timeframe=bt.TimeFrame.Minutes))
ce.addstrategy(GhiGio)
gio_bt = ce.run()[0].gio
cach["đưa thẳng index có múi giờ vào PandasData"] = pd.DatetimeIndex(gio_bt)

print(f"Thanh đầu tiên thật: {t.iloc[0]}\n")
for ten, idx in cach.items():
    lech = (idx[0] - khung_1p.index[0]) / pd.Timedelta(hours=1)
    print(f"{ten:<58} thanh đầu {idx[0]:%H:%M} · thanh cuối phiên {idx[N0 - 1]:%H:%M} · lệch {lech:+.0f} giờ")

Thanh đầu tiên thật: 2025-10-20 09:00:00+07:00

bt_vn.chuan_bi_frame (tz_convert VN → tz_localize(None))   thanh đầu 09:00 · thanh cuối phiên 14:45 · lệch +0 giờ
tz_convert('UTC').tz_localize(None)                        thanh đầu 02:00 · thanh cuối phiên 07:45 · lệch -7 giờ
.values                                                    thanh đầu 02:00 · thanh cuối phiên 07:45 · lệch -7 giờ
đưa thẳng index có múi giờ vào PandasData                  thanh đầu 02:00 · thanh cuối phiên 07:45 · lệch -7 giờ


Ba cách sai cho cùng một kết quả: mọi thanh lùi **7 tiếng** về giờ UTC — phiên
chạy từ 02:00 tới 07:45. Cách thứ tư đáng sợ nhất vì nó trông như không làm gì
sai: backtrader nhận index có múi giờ, **không báo lỗi**, và âm thầm đổi về UTC
bên trong (`date2num` trừ `utcoffset`).

Lệch 7 tiếng không đổi *ngày* của thanh nào (02:00–07:45 vẫn cùng ngày), không
đổi thứ tự, không đổi giá. Nó chỉ phá đúng những luật viết theo giờ — mà luật
quan trọng nhất của giao dịch trong phiên là "đóng hết trước ATC". Mục 9 chạy
chiến lược trên cả hai bản và đếm.

## 6 · `resampledata`: 1 phút → 5 phút → 15 phút

Gộp trong backtrader thay vì tải thêm: một feed 1 phút, và `cerebro.resampledata`
dựng khung lớn từ chính nó, nên hai khung không bao giờ lệch nhau.

finlens cũng trả sẵn `interval="5min"` — đó là **thước đo** để kiểm phép gộp.
Một chi tiết quyết định kết quả: finlens đánh nhãn thanh theo **phút bắt đầu**
(thanh 09:00 là 09:00:00–09:00:59), còn backtrader mặc định coi nhãn là **thời
điểm kết thúc**. Tham số `boundoff=1` dời biên gộp đi một thanh để bù lại.

In [15]:
m5_tho, _ = tai_intraday(HOM_NAY - dt.timedelta(days=30), HOM_NAY, interval="5min")
m5 = bt_vn.bo_phien_chua_dong(m5_tho, client, cot="time")
khung_5p_goc = bt_vn.chuan_bi_frame(m5, don_vi_gia=DON_VI)

class GhiThanhLon(bt.Strategy):
    # Ghi lại thanh của datas[1] mỗi khi nó có thanh mới
    def __init__(self):
        self.thanh, self._n = [], 0
    def next(self):
        d = self.datas[1]
        if len(d) != self._n:
            self._n = len(d)
            self.thanh.append((d.datetime.datetime(0), self.datas[0].datetime.datetime(0),
                               d.open[0], d.high[0], d.low[0], d.close[0], d.volume[0]))

def feed_1p(khung):
    return bt.feeds.PandasData(dataname=khung, openinterest=None,
                               timeframe=bt.TimeFrame.Minutes, compression=1)

def gop(khung, phut, boundoff):
    ce = bt.Cerebro(stdstats=False)
    f = feed_1p(khung)
    ce.adddata(f)
    ce.resampledata(f, timeframe=bt.TimeFrame.Minutes, compression=phut, boundoff=boundoff)
    ce.addstrategy(GhiThanhLon)
    cot = ["nhan", "luc_1p", "open", "high", "low", "close", "volume"]
    return pd.DataFrame(ce.run()[0].thanh, columns=cot).set_index("nhan")

doan_1p = khung_1p[khung_1p.index >= khung_5p_goc.index.min()]
OHLCV = ["open", "high", "low", "close", "volume"]

def so_thanh_khop(g: pd.DataFrame, goc: pd.DataFrame) -> int:
    # Thanh 5 phút của finlens có một thanh gộp y hệt (OHLCV) trong cùng phiên — không phụ thuộc cách đánh nhãn
    khoa = lambda k: k[OHLCV].round(4).assign(ngay=k.index.normalize())
    return len(khoa(goc).merge(khoa(g).drop_duplicates(), on=["ngay", *OHLCV]))

for bo in (0, 1):
    g = gop(doan_1p, 5, bo)
    print(f"boundoff={bo}: {len(g)} thanh 5 phút (finlens: {len(khung_5p_goc)}) · "
          f"khớp OHLCV {so_thanh_khop(g, khung_5p_goc)}/{len(khung_5p_goc)} "
          f"· thanh đầu nhãn {g.index[0]:%H:%M}, khối lượng {g['volume'].iloc[0]:,.0f} "
          f"(finlens {khung_5p_goc.index[0]:%H:%M}: {khung_5p_goc['volume'].iloc[0]:,.0f})")

C:\Program Files\Python312\Lib\asyncio\events.py:88: DataQualityWarning: [SESSION_IN_PROGRESS] Phiên ngày 2026-10-20 đang chạy (giờ phiên 09:00–15:00), nên khoảng bạn yêu cầu còn thay đổi: gọi lại đúng khoảng này sau 15:00 sẽ ra số khác. `meta.as_of` đang là 2026-10-20T11:15:00+07:00. Đừng cache kết quả này như thể nó đã chốt.
  self._context.run(self._callback, *self._args)


boundoff=0: 1029 thanh 5 phút (finlens: 1029) · khớp OHLCV 0/1029 · thanh đầu nhãn 09:00, khối lượng 4,549 (finlens 09:00: 8,544)


boundoff=1: 1029 thanh 5 phút (finlens: 1029) · khớp OHLCV 1029/1029 · thanh đầu nhãn 09:04, khối lượng 8,544 (finlens 09:00: 8,544)


Với `boundoff=0`, thanh 5 phút đầu tiên của mỗi phiên **chỉ chứa thanh 09:00**,
thanh thứ hai chứa 09:01–09:05… — mọi thanh lệch một phút so với finlens, và
"khoảng mở cửa 15 phút" của chiến lược sẽ thiếu đúng phút mở cửa. Với
`boundoff=1` thì khớp với chuỗi 5 phút của finlens; nhãn của backtrader khi đó là
**phút của thanh 1 phút cuối cùng** trong nhóm (09:04, 09:09…).

Thanh 15 phút cho chiến lược dùng đúng thiết lập đó:

In [16]:
g15 = gop(khung_1p.tail(CHUAN * 3), 15, 1)
mot_phien = g15[g15.index.normalize() == g15.index.normalize().max()]
print(f"{len(mot_phien)} thanh 15 phút trong phiên {mot_phien.index[0]:%Y-%m-%d}")
print("Thanh đầu (khoảng mở cửa) — có ở strategy lúc thanh 1 phút", f"{mot_phien['luc_1p'].iloc[0]:%H:%M}:")
print(mot_phien.iloc[:2].to_string())

fig = nen(mot_phien.reset_index(), x="nhan",
          tieu_de=f"{HD} — thanh 15 phút dựng bằng resampledata, phiên {mot_phien.index[0]:%d/%m/%Y}",
          phu_de="Nhãn = phút cuối của nhóm (boundoff=1) · thanh cuối là ATC 14:45 đứng riêng",
          nhan_gia="điểm chỉ số")
fig.update_yaxes(title_text="hợp đồng", row=2, col=1)
fig

17 thanh 15 phút trong phiên 2026-10-19
Thanh đầu (khoảng mở cửa) — có ở strategy lúc thanh 1 phút 09:14:
                                 luc_1p    open    high     low   close   volume
nhan                                                                            
2026-10-19 09:14:00 2026-10-19 09:14:00  1864.6  1870.0  1864.2  1868.5  15537.0
2026-10-19 09:29:00 2026-10-19 09:29:00  1868.4  1869.5  1865.2  1866.2  12648.0


## 7 · `replaydata`: một thanh ngày lớn dần từng phút

`resampledata` đưa thanh 15 phút cho strategy **khi nó đã xong**. `replaydata`
thì đưa thanh ngày **đang hình thành**: mỗi thanh 1 phút, `next()` được gọi lại
với cùng một thanh ngày, `high`/`low`/`close`/`volume` cập nhật dần, và `len(data)`
không đổi cho tới khi sang phiên mới. Hữu ích khi luật viết theo khung ngày
("giá vượt đỉnh hôm qua") nhưng phải phản ứng trong phiên.

In [17]:
class GhiReplay(bt.Strategy):
    def __init__(self):
        self.ghi = []
    def next(self):
        d = self.data
        self.ghi.append(dict(len=len(d), ngay=d.datetime.date(0), open=d.open[0], high=d.high[0],
                             low=d.low[0], close=d.close[0], volume=d.volume[0]))

doan = khung_1p[khung_1p.index.normalize() >= khung_1p.index.normalize().unique()[-20]]
ce = bt.Cerebro(stdstats=False)
ce.replaydata(feed_1p(doan), timeframe=bt.TimeFrame.Days, compression=1)
ce.addstrategy(GhiReplay)
rp = pd.DataFrame(ce.run()[0].ghi)

print(f"{len(doan):,} thanh 1 phút → next() được gọi {len(rp):,} lần · len(data) cuối = {rp['len'].iloc[-1]} thanh ngày")
ngay_cuoi = rp[rp["ngay"] == rp["ngay"].max()]
print(f"\nThanh ngày {ngay_cuoi['ngay'].iloc[0]} lớn dần (len(data) luôn = {ngay_cuoi['len'].iloc[0]}):")
print(ngay_cuoi.iloc[[0, 29, 149, len(ngay_cuoi) - 1]][["open", "high", "low", "close", "volume"]]
      .set_axis(["sau 1 phút", "sau 30 phút", "hết buổi sáng", "sau ATC"]).to_string())

4,818 thanh 1 phút → next() được gọi 4,818 lần · len(data) cuối = 20 thanh ngày

Thanh ngày 2026-10-19 lớn dần (len(data) luôn = 20):
                 open    high     low   close    volume
sau 1 phút     1864.6  1867.0  1864.2  1866.1    4815.0
sau 30 phút    1864.6  1870.0  1864.2  1866.2   28185.0
hết buổi sáng  1864.6  1870.0  1860.5  1861.4   77483.0
sau ATC        1864.6  1870.0  1841.0  1841.0  174122.0


In [18]:
# Thanh ngày cuối cùng của replay so với EOD
cuoi_ngay = rp.groupby("ngay").last()
cuoi_ngay.index = pd.DatetimeIndex(cuoi_ngay.index)
ss = cuoi_ngay.join(eod.set_index("date")[["open", "high", "low", "close", "volume"]], rsuffix="_eod", how="inner")
for c in ["open", "high", "low", "close", "volume"]:
    print(f"{c:<7} khớp EOD {int(np.isclose(ss[c], ss[c + '_eod']).sum())}/{len(ss)} phiên · "
          f"lệch tuyệt đối trung bình {(ss[c] - ss[c + '_eod']).abs().mean():,.2f}")

open    khớp EOD 17/20 phiên · lệch tuyệt đối trung bình 0.09
high    khớp EOD 20/20 phiên · lệch tuyệt đối trung bình 0.00
low     khớp EOD 20/20 phiên · lệch tuyệt đối trung bình 0.00
close   khớp EOD 20/20 phiên · lệch tuyệt đối trung bình 0.00
volume  khớp EOD 0/20 phiên · lệch tuyệt đối trung bình 3,109.35


Cao, thấp, đóng cửa của thanh ngày dựng từ 1 phút khớp EOD; giá mở cửa lệch ở
vài phiên, còn **khối lượng thì không khớp phiên nào**. Khối lượng EOD và tổng
khối lượng 1 phút là hai phép đo khác nhau (có thể vì giao dịch thoả thuận
8:45–14:45 và khớp ATO không nằm gọn trong các thanh 1 phút — finlens không tách
các phần này, nên đây là giả thuyết, không phải kết luận). Hệ quả thực tế: một luật dùng khối
lượng ngày sẽ cho kết quả khác nhau tuỳ bạn lấy khối lượng từ đâu.

## 8 · Chiến lược trong phiên: phá vỡ khoảng mở cửa, long **và** short

Luật (chọn để minh hoạ cơ chế, **không** phải khuyến nghị):

- **Khoảng mở cửa** = thanh 15 phút đầu tiên của phiên (09:00–09:14)
- Thanh 15 phút **đóng cửa trên** đỉnh khoảng → mua 1 HĐ; **dưới** đáy → bán khống 1 HĐ.
  Tối đa một lần vào lệnh mỗi phiên, không vào sau 14:00
- Cắt lỗ: giá 1 phút đóng cửa vượt cạnh đối diện của khoảng
- **Từ 14:25 đóng mọi vị thế** — lệnh thị trường khớp ở thanh 1 phút kế tiếp, trước ATC
- Chốt chặn: nếu sang phiên mới mà còn vị thế (phiên trước thiếu thanh chiều) thì đóng ngay và đếm lại

Strategy nhìn hai feed: `datas[0]` là 1 phút (để khớp lệnh và kiểm giờ),
`datas[1]` là 15 phút từ `resampledata` (để ra tín hiệu).

In [19]:
class PhaVoMoCua(bt.Strategy):
    params = (("gio_dong", GIO_DONG_VI_THE), ("gio_cuoi_vao", GIO_CUOI_VAO_LENH), ("so_hd", 1))

    def __init__(self):
        self.m1, self.m15 = self.datas
        self._n15, self._ngay, self.khoang, self.da_vao, self.lenh = 0, None, None, False, None
        self.dem = dict(dong_truoc_atc=0, cat_lo=0, qua_dem=0)
        self.khop, self.vong, self.ky_quy_max, self.tong_phi = [], [], 0.0, 0.0

    def notify_order(self, o):
        if o.status == o.Completed:
            self.khop.append(dict(luc=bt.num2date(o.executed.dt), size=o.executed.size,
                                  gia=o.executed.price, phi=o.executed.comm, ky_quy=o.executed.value))
            self.tong_phi += o.executed.comm
            self.ky_quy_max = max(self.ky_quy_max, o.executed.value)
        if o.status in (o.Completed, o.Canceled, o.Margin, o.Rejected):
            self.lenh = None

    def notify_trade(self, t):
        if t.isclosed:
            self.vong.append(dict(mo=bt.num2date(t.dtopen), dong=bt.num2date(t.dtclose),
                                  chieu="long" if t.long else "short", pnl=t.pnl, pnlcomm=t.pnlcomm))

    def next(self):
        luc = self.m1.datetime.datetime(0)
        if luc.date() != self._ngay:                       # sang phiên mới
            self._ngay, self.khoang, self.da_vao = luc.date(), None, False
            if self.position and not self.lenh:
                self.dem["qua_dem"] += 1
                self.lenh = self.close()
                return
        moi15 = len(self.m15) != self._n15
        if moi15:
            self._n15 = len(self.m15)
            if self.khoang is None and self.m15.datetime.date(0) == self._ngay:
                self.khoang = (self.m15.high[0], self.m15.low[0])
                return
        if self.lenh:
            return

        vt = self.position.size
        if vt:
            if luc.time() >= self.p.gio_dong:
                self.dem["dong_truoc_atc"] += 1
                self.lenh = self.close()
            elif self.khoang and ((vt > 0 and self.m1.close[0] < self.khoang[1])
                                  or (vt < 0 and self.m1.close[0] > self.khoang[0])):
                self.dem["cat_lo"] += 1
                self.lenh = self.close()
            return

        if self.khoang is None or self.da_vao or not moi15 or luc.time() >= self.p.gio_cuoi_vao:
            return
        dong_15 = self.m15.close[0]
        if dong_15 > self.khoang[0]:
            self.lenh, self.da_vao = self.buy(size=self.p.so_hd), True
        elif dong_15 < self.khoang[1]:
            self.lenh, self.da_vao = self.sell(size=self.p.so_hd), True


def chay_trong_phien(khung):
    ce = bt.Cerebro(stdstats=False)
    f = feed_1p(khung)
    ce.adddata(f)
    ce.resampledata(f, timeframe=bt.TimeFrame.Minutes, compression=15, boundoff=1)
    ce.broker.setcash(VON)
    gan_phi(ce)
    ce.addstrategy(PhaVoMoCua)
    bt_vn.them_analyzer(ce)
    return ce.run()[0]

t0 = time.perf_counter()
s_orb = chay_trong_phien(khung_1p)
kq_orb = bt_vn.ket_qua(s_orb)
print(f"Chạy {len(khung_1p):,} thanh 1 phút trong {time.perf_counter() - t0:.0f} giây")
print(f"Đếm sự kiện: {s_orb.dem}")
vong_orb = pd.DataFrame(s_orb.vong)
print(f"{len(vong_orb)} vòng: {(vong_orb['chieu'] == 'long').sum()} long · {(vong_orb['chieu'] == 'short').sum()} short · "
      f"tỷ lệ thắng {kq_orb['tom_tat']['ty_le_thang']:.1%}")
print(f"Tiền cuối {s_orb.broker.getvalue():,.0f} VND · tổng phí {s_orb.tong_phi:,.0f} VND · "
      f"ký quỹ lớn nhất {s_orb.ky_quy_max:,.0f} VND")

Chạy 58,520 thanh 1 phút trong 28 giây
Đếm sự kiện: {'dong_truoc_atc': 144, 'cat_lo': 89, 'qua_dem': 1}
234 vòng: 106 long · 128 short · tỷ lệ thắng 44.0%
Tiền cuối 115,230,000 VND · tổng phí 4,680,000 VND · ký quỹ lớn nhất 42,004,000 VND


### Đối chiếu một vòng giao dịch bằng tay

Lấy vòng đầu tiên: lãi/lỗ phải bằng `Δđiểm × 100.000 × chiều − 2 × phí`, ký quỹ
phong toả phải bằng `giá vào × 100.000 × 20%`.

In [20]:
v0 = vong_orb.iloc[0]
vao, ra = s_orb.khop[0], s_orb.khop[1]
chieu = np.sign(vao["size"])
lai_tay = (ra["gia"] - vao["gia"]) * HE_SO_NHAN * chieu - vao["phi"] - ra["phi"]
print(f"Vòng {v0['chieu']}: vào {vao['luc']:%Y-%m-%d %H:%M} @ {vao['gia']:.1f} · ra {ra['luc']:%H:%M} @ {ra['gia']:.1f}")
print(f"Lãi/lỗ   — backtrader {v0['pnlcomm']:,.0f} · tay ({ra['gia']:.1f} − {vao['gia']:.1f}) × 100.000 × "
      f"{chieu:+.0f} − {vao['phi'] + ra['phi']:,.0f} = {lai_tay:,.0f} VND")
print(f"Ký quỹ   — backtrader {vao['ky_quy']:,.0f} · tay {vao['gia']:.1f} × 100.000 × {TY_LE_KY_QUY:.0%} = "
      f"{vao['gia'] * HE_SO_NHAN * TY_LE_KY_QUY:,.0f} VND")
print(f"Tổng lãi/lỗ các vòng {vong_orb['pnlcomm'].sum():,.0f} = tiền cuối − vốn {s_orb.broker.getvalue() - VON:,.0f} VND")

gio_ra = pd.Series([k["luc"] for k in s_orb.khop[1::2]])
print(f"\nGiờ khớp lệnh đóng muộn nhất trong phiên: {gio_ra.dt.time.max()} (ATC bắt đầu {GIO_ATC})")

RA_FILE = GOC / "output" / f"giao_dich_orb_{HD}_{HOM_NAY:%Y%m%d}.csv"
RA_FILE.parent.mkdir(exist_ok=True)
vong_orb.to_csv(RA_FILE, index=False)
print(f"Nhật ký vòng giao dịch → {RA_FILE.relative_to(GOC)}")

Vòng short: vào 2025-10-20 10:00 @ 1954.7 · ra 14:26 @ 1881.9
Lãi/lỗ   — backtrader 7,260,000 · tay (1881.9 − 1954.7) × 100.000 × -1 − 20,000 = 7,260,000 VND
Ký quỹ   — backtrader 39,094,000 · tay 1954.7 × 100.000 × 20% = 39,094,000 VND
Tổng lãi/lỗ các vòng 15,230,000 = tiền cuối − vốn 15,230,000 VND

Giờ khớp lệnh đóng muộn nhất trong phiên: 14:26:00 (ATC bắt đầu 14:30:00)
Nhật ký vòng giao dịch → output\giao_dich_orb_VN30F1M_20261020.csv


## 9 · ⚠️ Cạm bẫy (a) bằng số: cùng chiến lược, thanh lệch 7 tiếng

In [21]:
khung_sai = khung_1p.copy()
khung_sai.index = cach[".values"]
assert (cach[".values"] == cach["tz_convert('UTC').tz_localize(None)"]).all()
assert (cach[".values"][:N0] == cach["đưa thẳng index có múi giờ vào PandasData"]).all()

s_sai = chay_trong_phien(khung_sai)
kq_sai = bt_vn.ket_qua(s_sai)

def dong_ket_qua(ten, s, kq):
    tt = kq["tom_tat"]
    return {"bản": ten, "đóng trước ATC": s.dem["dong_truoc_atc"], "cắt lỗ": s.dem["cat_lo"],
            "giữ qua đêm": s.dem["qua_dem"], "số vòng": tt["so_vong_dong"],
            "lãi/lỗ (tr VND)": (s.broker.getvalue() - VON) / 1e6, "CAGR": tt["cagr"], "MDD": tt["mdd"],
            "Sharpe": tt["sharpe"]}

so_sanh_tz = pd.DataFrame([dong_ket_qua("đúng — bt_vn.chuan_bi_frame", s_orb, kq_orb),
                           dong_ket_qua("sai — .values / UTC", s_sai, kq_sai)]).set_index("bản")
print(so_sanh_tz.round(3).to_string())

                             đóng trước ATC  cắt lỗ  giữ qua đêm  số vòng  lãi/lỗ (tr VND)   CAGR    MDD  Sharpe
bản                                                                                                             
đúng — bt_vn.chuan_bi_frame             144      89            1      234            15.23  0.157 -0.130   0.800
sai — .values / UTC                       0      96          138      234             2.87  0.030 -0.164   0.243


In [22]:
fig = go.Figure()
for i, (ten, kq) in enumerate([("Giờ Việt Nam (đúng)", kq_orb), ("Lệch 7 tiếng (sai)", kq_sai)]):
    fig.add_trace(go.Scatter(x=kq["duong_von"].index, y=kq["duong_von"] * VON / 1e6, name=ten,
                             line=dict(width=2, color=CHUOI[i])))
fig.update_layout(title_text=f"{HD} — phá vỡ khoảng mở cửa, 1 hợp đồng<br>"
                  "<sub style='color:#52514e'>Cùng dữ liệu, cùng luật; chỉ khác cách bỏ múi giờ của cột time</sub>",
                  yaxis_title="vốn, triệu VND", height=460, hovermode="x unified")
fig

Bản sai **không đóng vị thế trước ATC lần nào**: điều kiện `giờ ≥ 14:25` không
bao giờ đúng với các thanh chạy từ 02:00 tới 07:45. Mọi vị thế còn mở cuối phiên
bị mang qua đêm, tới đầu phiên sau mới bị chốt chặn đóng — tức chiến
lược "trong phiên" thật ra đang ôm rủi ro gap qua đêm, ký quỹ bị VSDC tính, và kết
quả không còn nói gì về luật được viết ra. Không có lỗi, không có cảnh báo; chỉ
có con số khác.

## 10 · ⚠️ Cạm bẫy (b): chuỗi nối `VN30F1M` nhảy ở ngày đáo hạn

`VN30F1M` không phải một hợp đồng. Đến hết ngày giao dịch cuối cùng (thứ Năm thứ
ba), hợp đồng tháng này đáo hạn và phiên sau `VN30F1M` trỏ sang hợp đồng tháng
kế tiếp — vốn đang giao dịch với tên `VN30F2M`. Hai hợp đồng có basis khác nhau,
nên chuỗi nối có một **bước nhảy không ai kiếm được** ở mỗi lần cuộn.

Bước 1: dựng lịch đáo hạn từ quy tắc, dùng lịch phiên EOD để xử lý ngày nghỉ.

In [23]:
def ngay_dao_han(lich: pd.DatetimeIndex) -> pd.DatetimeIndex:
    # Thứ Năm thứ ba của mỗi tháng; trùng ngày nghỉ thì lùi về phiên liền trước (trong cùng tháng)
    ra = []
    for thang in pd.period_range(lich.min(), lich.max(), freq="M"):
        thu5 = pd.date_range(thang.start_time, thang.end_time.normalize(), freq="W-THU")[2]
        truoc = lich[(lich <= thu5) & (lich.to_period("M") == thang)]
        if len(truoc):
            ra.append(truoc[-1])
    return pd.DatetimeIndex(ra)

BAT_DAU_EOD = lui_ngay(HOM_NAY, nam=6)
f2 = bt_vn.bo_phien_chua_dong(client.eod.derivative.ohlcv("VN30F2M", start=BAT_DAU_EOD), client)
vn30 = bt_vn.bo_phien_chua_dong(client.eod.index.ohlcv("VN30", start=BAT_DAU_EOD), client)
basis = bt_vn.bo_phien_chua_dong(client.eod.derivative.basis(HD, start=BAT_DAU_EOD), client)

d = (eod.set_index("date")[["open", "high", "low", "close"]]
     .join(f2.set_index("date")["close"].rename("f2"))
     .join(vn30.set_index("date")["close"].rename("vn30"))
     .join(basis.set_index("date")["basis"]))
d = d.dropna()
lich = d.index
DAO_HAN = ngay_dao_han(lich)
SAU_DAO_HAN = pd.DatetimeIndex([lich[lich.get_loc(x) + 1] for x in DAO_HAN if lich.get_loc(x) + 1 < len(lich)])
print(f"{len(lich)} phiên từ {lich.min():%Y-%m-%d} (bỏ {len(eod) - len(lich)} phiên thiếu VN30F2M/VN30/basis) · "
      f"{len(DAO_HAN)} ngày đáo hạn")
print("Ngày đáo hạn KHÔNG rơi vào thứ Năm (do nghỉ lễ):", [f"{x:%Y-%m-%d} ({x:%a})" for x in DAO_HAN if x.weekday() != 3])

1496 phiên từ 2020-10-19 (bỏ 0 phiên thiếu VN30F2M/VN30/basis) · 72 ngày đáo hạn
Ngày đáo hạn KHÔNG rơi vào thứ Năm (do nghỉ lễ): ['2024-04-17 (Wed)', '2026-02-13 (Fri)']


Bước 2: kiểm quy tắc bằng dữ liệu. Nếu chuỗi nối đổi hợp đồng ở phiên **sau**
ngày đáo hạn, bước nhảy của basis (`client.eod.derivative.basis`) phải lớn nhất
đúng ở phiên đó, và `VN30F1M` hôm sau phải nối tiếp `VN30F2M` hôm trước.

In [24]:
d["d_basis"] = d["basis"].diff()
d["d_f1"], d["d_vn30"] = d["close"].diff(), d["vn30"].diff()
vi_tri = {x: i for i, x in enumerate(lich)}
hang = []
for k in (-2, -1, 0, 1, 2, 3):
    ii = [vi_tri[x] + k for x in DAO_HAN if 0 <= vi_tri[x] + k < len(lich)]
    hang.append({"phiên so với đáo hạn": f"{k:+d}", "|Δbasis| trung bình (điểm)": d["d_basis"].iloc[ii].abs().mean()})
thuong = d["d_basis"].drop(SAU_DAO_HAN).abs().mean()
print(pd.DataFrame(hang).round(2).to_string(index=False))
print(f"Mọi phiên khác: {thuong:.2f} điểm")

# Hợp đồng nối tiếp: F1M(t) so với F2M(t−1), trừ phần VN30 đi trong phiên
noi = (d["close"] - d["f2"].shift(1) - d["d_vn30"]).loc[SAU_DAO_HAN].abs()
print(f"\nPhiên sau đáo hạn: |ΔF1M − ΔVN30| = {d['d_basis'].loc[SAU_DAO_HAN].abs().mean():.2f} điểm, "
      f"nhưng |F1M(t) − F2M(t−1) − ΔVN30| = {noi.mean():.2f} điểm")
print(f"Basis ngày đáo hạn: |basis| trung bình {d['basis'].loc[DAO_HAN].abs().mean():.2f} điểm "
      f"(mọi phiên: {d['basis'].abs().mean():.2f})")

phiên so với đáo hạn  |Δbasis| trung bình (điểm)
                  -2                        2.99
                  -1                        4.13
                  +0                        4.34
                  +1                        6.34
                  +2                        4.48
                  +3                        3.81
Mọi phiên khác: 3.99 điểm

Phiên sau đáo hạn: |ΔF1M − ΔVN30| = 6.34 điểm, nhưng |F1M(t) − F2M(t−1) − ΔVN30| = 4.71 điểm
Basis ngày đáo hạn: |basis| trung bình 2.97 điểm (mọi phiên: 4.71)


Bước nhảy của basis cao nhất ở phiên **+1** — đúng phiên chuỗi nối đổi hợp đồng.
Và ở phiên đó, `VN30F1M` nối tiếp `VN30F2M` của hôm trước sát hơn là nối tiếp
chính nó — gần với mức của một phiên thường: hai "giá" liên tiếp trong chuỗi nối
là giá của hai hợp đồng khác nhau. Phần còn lệch là nhiễu của `VN30F2M`, hợp
đồng ít thanh khoản hơn nhiều.
Basis ngày đáo hạn nhỏ hơn bình thường nhưng không về 0, vì giá thanh toán cuối
cùng là trung bình 30 phút cuối chứ không phải giá đóng cửa của VN30.

Bước 3: đo bước nhảy. **Chênh lệch cuộn** = `VN30F2M − VN30F1M` lúc đóng cửa ngày
đáo hạn — khoảng cách giữa hợp đồng cũ và hợp đồng sắp thay nó.

In [25]:
gap = (d["f2"] - d["close"]).loc[DAO_HAN].rename("gap")
gap_sau = pd.Series(gap.to_numpy()[: len(SAU_DAO_HAN)], index=SAU_DAO_HAN)
print(f"Chênh lệch cuộn: trung bình |gap| {gap.abs().mean():.2f} điểm · lớn nhất {gap.abs().max():.1f} điểm "
      f"({gap.abs().idxmax():%Y-%m-%d}) · tổng cộng dồn {gap.sum():+.1f} điểm")
print(f"Tương quan gap với Δbasis phiên sau: {np.corrcoef(gap_sau, d['d_basis'].loc[SAU_DAO_HAN])[0, 1]:.2f}")
print(f"So với biến động: độ lệch chuẩn thay đổi ngày của VN30F1M {d['d_f1'].std():.1f} điểm")
print(f"Một hợp đồng: |gap| trung bình = {gap.abs().mean() * HE_SO_NHAN:,.0f} VND mỗi lần cuộn")

g24 = gap.tail(24).rename_axis("ngay").reset_index()
g24["nhan"] = g24["ngay"].dt.strftime("%m/%y")
thanh_doi_mau(g24, x="nhan", y="gap",
              tieu_de=f"Chênh lệch cuộn VN30F2M − VN30F1M ở 24 ngày đáo hạn gần nhất",
              phu_de="Bước nhảy chuỗi nối VN30F1M mang vào phiên sau đáo hạn · âm = hợp đồng kế tiếp rẻ hơn",
              nhan_y="điểm chỉ số", dinh_dang_nhan="{:+.1f}")

Chênh lệch cuộn: trung bình |gap| 8.11 điểm · lớn nhất 30.3 điểm (2021-01-21) · tổng cộng dồn -134.7 điểm
Tương quan gap với Δbasis phiên sau: 0.74
So với biến động: độ lệch chuẩn thay đổi ngày của VN30F1M 19.4 điểm
Một hợp đồng: |gap| trung bình = 810,972 VND mỗi lần cuộn


Bước 4: bước nhảy sinh tín hiệu giả thế nào. Tín hiệu "cú sốc" của mục 11 là
`z = Δclose / độ lệch chuẩn 20 phiên trước`, kích hoạt khi `|z| ≥ 2`. Trên một
chuỗi nối thô, bước nhảy cuộn cộng thẳng vào `Δclose` của phiên sau đáo hạn.

Cách xử lý: **điều chỉnh lùi theo chênh lệch cuộn** — cộng `gap` của mỗi lần cuộn
vào mọi giá *trước* nó. Chuỗi đã điều chỉnh có giá hiện tại đúng bằng giá thật,
và mỗi đoạn giữa hai lần cuộn giữ nguyên khoảng cách điểm của hợp đồng tương ứng —
đúng với lãi/lỗ của người giữ vị thế rồi cuộn sang hợp đồng mới.

In [26]:
def dieu_chinh_cuon(khung: pd.DataFrame, gap: pd.Series) -> pd.DataFrame:
    # Cộng dồn gap của mọi lần cuộn từ ngày đó về sau (ngày đáo hạn vẫn là hợp đồng cũ → được cộng)
    bu = pd.Series(0.0, index=khung.index)
    for ngay, g in gap.items():
        bu[bu.index <= ngay] += g
    ra = khung.copy()
    ra[["open", "high", "low", "close"]] = ra[["open", "high", "low", "close"]].add(bu, axis=0)
    return ra

d_dc = dieu_chinh_cuon(d, gap)
lech_gia = (d_dc["close"] - d["close"]).abs()
print(f"Chuỗi điều chỉnh lệch giá thật tối đa {lech_gia.max():.1f} điểm "
      f"({(lech_gia / d['close']).max():.1%} giá thô, ở {(lech_gia / d['close']).idxmax():%Y-%m-%d}); "
      f"sau lần cuộn cuối ({DAO_HAN.max():%Y-%m-%d}) lệch {lech_gia[lech_gia.index > DAO_HAN.max()].max():.1f} điểm")
e_tho = (d["close"].diff() - d["d_vn30"]).abs()
e_dc = (d_dc["close"].diff() - d["d_vn30"]).abs()
print(f"|ΔF1M − ΔVN30| phiên sau đáo hạn: thô {e_tho.loc[SAU_DAO_HAN].mean():.2f} → điều chỉnh "
      f"{e_dc.loc[SAU_DAO_HAN].mean():.2f} điểm (phiên thường {e_tho.drop(SAU_DAO_HAN).mean():.2f})")

NGUONG = 2.0
z_t = z_thay_doi(d["close"])
z_d = z_thay_doi(d_dc["close"])
tin_t, tin_d = np.sign(z_t) * (z_t.abs() >= NGUONG), np.sign(z_d) * (z_d.abs() >= NGUONG)
ty_le_nen = (tin_d.drop(SAU_DAO_HAN) != 0).mean()
print(f"\nTín hiệu |z| ≥ {NGUONG}: chuỗi thô {int((tin_t != 0).sum())} · đã điều chỉnh {int((tin_d != 0).sum())}")
print(f"Trên {len(SAU_DAO_HAN)} phiên sau đáo hạn: thô {int((tin_t.loc[SAU_DAO_HAN] != 0).sum())} · "
      f"đã điều chỉnh {int((tin_d.loc[SAU_DAO_HAN] != 0).sum())} · kỳ vọng theo tỷ lệ nền "
      f"{ty_le_nen:.1%} → {ty_le_nen * len(SAU_DAO_HAN):.1f}")
khac = tin_t != tin_d
print(f"Phiên có tín hiệu khác nhau giữa hai chuỗi: {int(khac.sum())}, trong đó {int(khac.loc[SAU_DAO_HAN].sum())} "
      f"đúng phiên sau đáo hạn (phần còn lại: độ lệch chuẩn 20 phiên mang bước nhảy theo suốt 20 phiên)")

long_tho = d["close"].iloc[-1] - d["close"].iloc[0]
long_dc = d_dc["close"].iloc[-1] - d_dc["close"].iloc[0]
print(f"\nLong 1 HĐ và cuộn đều từ {lich.min():%Y-%m-%d}: chuỗi thô nói {long_tho:+,.1f} điểm, "
      f"thực tế khi cuộn {long_dc:+,.1f} điểm → lệch {(long_dc - long_tho) * HE_SO_NHAN / 1e6:+,.1f} triệu VND/HĐ "
      f"(VN30 cùng kỳ {d['vn30'].iloc[-1] - d['vn30'].iloc[0]:+,.1f} điểm)")

Chuỗi điều chỉnh lệch giá thật tối đa 179.0 điểm (17.8% giá thô, ở 2021-01-28); sau lần cuộn cuối (2026-10-15) lệch 0.0 điểm
|ΔF1M − ΔVN30| phiên sau đáo hạn: thô 6.34 → điều chỉnh 4.71 điểm (phiên thường 3.99)

Tín hiệu |z| ≥ 2.0: chuỗi thô 151 · đã điều chỉnh 151
Trên 72 phiên sau đáo hạn: thô 15 · đã điều chỉnh 8 · kỳ vọng theo tỷ lệ nền 10.0% → 7.2
Phiên có tín hiệu khác nhau giữa hai chuỗi: 35, trong đó 7 đúng phiên sau đáo hạn (phần còn lại: độ lệch chuẩn 20 phiên mang bước nhảy theo suốt 20 phiên)

Long 1 HĐ và cuộn đều từ 2020-10-19: chuỗi thô nói +937.6 điểm, thực tế khi cuộn +1,072.3 điểm → lệch +13.5 triệu VND/HĐ (VN30 cùng kỳ +935.7 điểm)


## 11 · Chiến lược EOD: đảo chiều sau cú sốc, chuỗi thô vs chuỗi đã điều chỉnh

Luật: phiên có `z ≤ −2` → mua 1 HĐ ở giá mở cửa phiên sau; `z ≥ 2` → bán khống;
giữ 3 phiên rồi đóng. `z` tính ngay trong backtrader bằng `bt.ind.StdDev` — cùng
công thức với `z_thay_doi` ở trên. Chiến lược này **giữ qua đêm**, nên cũng là
chỗ để đo lại lỗi `getvalue()` của mục 3.

In [27]:
class DaoChieuEOD(bt.Strategy):
    params = (("nguong", NGUONG), ("giu", 3), ("cua_so", 20), ("so_hd", 1))

    def __init__(self):
        delta = self.data.close - self.data.close(-1)
        self.z = delta / bt.ind.StdDev(delta, period=self.p.cua_so)(-1)
        self.lenh, self.vao, self.tong_phi, self.ky_quy_max = None, None, 0.0, 0.0
        self.ngay_giu = []   # phiên đóng cửa khi còn vị thế = một đêm giữ vị thế
        self.tu_choi = 0     # lệnh bị broker từ chối (Margin/Rejected)

    def notify_order(self, o):
        if o.status == o.Completed:
            self.tong_phi += o.executed.comm
            self.ky_quy_max = max(self.ky_quy_max, o.executed.value)
            self.vao = len(self) if self.position else None
        if o.status in (o.Margin, o.Rejected):
            self.tu_choi += 1
        if o.status in (o.Completed, o.Canceled, o.Margin, o.Rejected):
            self.lenh = None

    def next(self):
        if self.position:
            self.ngay_giu.append(pd.Timestamp(self.data.datetime.date(0)))
        if self.lenh:
            return
        if self.position:
            if len(self) - self.vao >= self.p.giu:
                self.lenh = self.close()
            return
        if self.z[0] <= -self.p.nguong:
            self.lenh = self.buy(size=self.p.so_hd)
        elif self.z[0] >= self.p.nguong:
            self.lenh = self.sell(size=self.p.so_hd)


def khung_eod(dd: pd.DataFrame) -> pd.DataFrame:
    return dd[["open", "high", "low", "close"]].assign(volume=eod.set_index("date")["volume"]).reset_index()

def chay_eod(dd, cach="dung"):
    ce = bt.Cerebro(stdstats=False)
    ce.adddata(bt_vn.feed_tu_finlens(khung_eod(dd), don_vi_gia=DON_VI, name=HD))
    ce.broker.setcash(VON)
    gan_phi(ce, cach)
    ce.addstrategy(DaoChieuEOD)
    bt_vn.them_analyzer(ce)
    return ce.run()[0]

kq_eod = {}
for ten, dd, cach_phi in [("thô · PhiPhaiSinhCu", d, "cu"),
                          ("thô", d, "dung"),
                          ("điều chỉnh cuộn", d_dc, "dung")]:
    s = chay_eod(dd, cach_phi)
    kq_eod[ten] = (s, bt_vn.ket_qua(s))

def dong_eod(ten, s, kq):
    tt = kq["tom_tat"]
    return {"bản": ten, "số vòng": tt["so_vong_dong"], "lãi/lỗ (tr VND)": (s.broker.getvalue() - VON) / 1e6,
            "CAGR": tt["cagr"], "MDD": tt["mdd"], "Sharpe": tt["sharpe"], "biến động": tt["bien_dong"]}

bang_eod = pd.DataFrame([dong_eod(t, *v) for t, v in kq_eod.items()]).set_index("bản")
print(bang_eod.round(4).to_string())

                     số vòng  lãi/lỗ (tr VND)    CAGR     MDD  Sharpe  biến động
bản                                                                             
thô · PhiPhaiSinhCu       89           -10.58 -0.0187 -0.3394 -0.0140     0.1797
thô                       89           -10.58 -0.0187 -0.3420 -0.0403     0.1574
điều chỉnh cuộn           87             1.30  0.0022 -0.2214  0.0912     0.1538


In [28]:
s_t, kq_t = kq_eod["thô"]
s_d, kq_d = kq_eod["điều chỉnh cuộn"]
gd_t, gd_d = kq_t["giao_dich"], kq_d["giao_dich"]
vao_t = gd_t.iloc[::2]["date"].dt.normalize()
vao_d = gd_d.iloc[::2]["date"].dt.normalize()
# Lệnh vào khớp ở phiên sau tín hiệu → tín hiệu ở phiên sau đáo hạn thì khớp ở phiên +2
sau2 = pd.DatetimeIndex([lich[lich.get_loc(x) + 1] for x in SAU_DAO_HAN if lich.get_loc(x) + 1 < len(lich)])
print(f"Lệnh vào từ tín hiệu của phiên sau đáo hạn: thô {int(vao_t.isin(sau2).sum())} · "
      f"điều chỉnh {int(vao_d.isin(sau2).sum())}")
print(f"Lệnh bị từ chối vì ký quỹ: thô {s_t.tu_choi} · điều chỉnh {s_d.tu_choi}")
print(f"Lãi/lỗ chênh giữa hai bản: {(s_d.broker.getvalue() - s_t.broker.getvalue()) / 1e6:+,.2f} triệu VND "
      f"trên {kq_t['tom_tat']['so_vong_dong']} / {kq_d['tom_tat']['so_vong_dong']} vòng")

s_g, kq_g = kq_eod["thô · PhiPhaiSinhCu"]
von_g, von_v = kq_g["duong_von"] * VON, kq_t["duong_von"] * VON
print(f"\nLỗi getvalue() bản cũ trên chiến lược giữ qua đêm: đường vốn lệch tối đa {(von_g - von_v).abs().max():,.0f} VND, "
      f"MDD {kq_g['tom_tat']['mdd']:.2%} vs {kq_t['tom_tat']['mdd']:.2%}, tiền cuối lệch "
      f"{s_g.broker.getvalue() - s_t.broker.getvalue():,.0f} VND")

Lệnh vào từ tín hiệu của phiên sau đáo hạn: thô 14 · điều chỉnh 7
Lệnh bị từ chối vì ký quỹ: thô 0 · điều chỉnh 0
Lãi/lỗ chênh giữa hai bản: +11.88 triệu VND trên 89 / 87 vòng

Lỗi getvalue() bản cũ trên chiến lược giữ qua đêm: đường vốn lệch tối đa 3,520,000 VND, MDD -33.94% vs -34.20%, tiền cuối lệch 0 VND


Hai bảng trên nói hai điều khác nhau:

- **Chuỗi thô vs đã điều chỉnh**: số lệnh vào sinh ra từ bước nhảy cuộn, và phần
  lãi/lỗ của những vòng giữ vị thế *qua* ngày cuộn — ở chuỗi thô, một vị thế giữ
  qua ngày cuộn "kiếm" hoặc "mất" đúng bằng `gap` mà không ai trên thị trường trả
  cho bạn. Cách xử lý khác là **không giao dịch quanh ngày đáo hạn** (đóng hết
  trước, bỏ tín hiệu của phiên sau) — đơn giản hơn nhưng bỏ đi khoảng 2 trong mỗi ~21 phiên.
- **`PhiPhaiSinhCu` vs `gan_phi_phai_sinh`**: tiền cuối giống hệt, nhưng đường vốn — và mọi
  chỉ số tính từ nó như MDD, Sharpe — lệch trong lúc giữ vị thế.

Chuỗi đã điều chỉnh có một giới hạn: giá trong quá khứ không còn là giá thật, nên
**ký quỹ** (tỷ lệ theo giá) tính trên nó lệch theo đúng tỷ lệ lệch giá in ở dòng
đầu output của bước 4 mục 10 — không nhỏ. Với 1 hợp đồng và vốn dư dả thì
không lệnh nào bị từ chối vì ký quỹ (output trên), nên kết quả không đổi; với
chiến lược dùng gần hết ký quỹ thì phải tính ký quỹ theo giá thô.

## 12 · Trong phiên vs EOD, cùng một khoảng thời gian

Chiến lược EOD chạy trên 6 năm để có đủ ngày đáo hạn; để so với chiến lược
trong phiên, cắt lợi suất ngày của nó về đúng khoảng có dữ liệu 1 phút.

In [29]:
tu = kq_orb["loi_suat"].index.min().normalize()
ls_eod = kq_d["loi_suat"][kq_d["loi_suat"].index >= tu]
so_vong_eod = int((kq_d["giao_dich"]["date"] >= tu).sum() // 2)
phi_eod = so_vong_eod * 2 * PHI_HD
dem_eod = int(sum(x >= tu for x in s_d.ngay_giu))

def dong_ss(ten, ls, so_vong, phi, qua_dem, ky_quy):
    tt = bt_vn.chi_so_hieu_qua(ls)
    return {"chiến lược": ten, "số vòng": so_vong, "phí (tr VND)": phi / 1e6,
            "lợi suất tổng": (1 + ls).prod() - 1, "CAGR": tt["cagr"], "MDD": tt["mdd"], "Sharpe": tt["sharpe"],
            "đêm giữ vị thế": qua_dem, "ký quỹ max (tr)": ky_quy / 1e6}

so_sanh = pd.DataFrame([
    dong_ss("Phá vỡ mở cửa (1 phút + 15 phút)", kq_orb["loi_suat"], len(vong_orb), s_orb.tong_phi,
            s_orb.dem["qua_dem"], s_orb.ky_quy_max),
    dong_ss("Đảo chiều sau cú sốc (EOD, đã điều chỉnh)", ls_eod, so_vong_eod, phi_eod,
            dem_eod, s_d.ky_quy_max),
]).set_index("chiến lược")
print(f"Khoảng so sánh: {tu:%Y-%m-%d} → {kq_orb['loi_suat'].index.max():%Y-%m-%d} · vốn {VON / 1e6:,.0f} triệu VND · 1 HĐ\n")
print(so_sanh.round(3).to_string())

Khoảng so sánh: 2025-10-20 → 2026-10-19 · vốn 100 triệu VND · 1 HĐ

                                           số vòng  phí (tr VND)  lợi suất tổng   CAGR    MDD  Sharpe  đêm giữ vị thế  ký quỹ max (tr)
chiến lược                                                                                                                            
Phá vỡ mở cửa (1 phút + 15 phút)               234          4.68          0.152  0.157 -0.130   0.800               1           42.004
Đảo chiều sau cú sốc (EOD, đã điều chỉnh)       11          0.22         -0.102 -0.104 -0.214  -0.511              44           39.954


In [30]:
fig = go.Figure()
for i, (ten, ls) in enumerate([("Phá vỡ mở cửa — trong phiên", kq_orb["loi_suat"]),
                               ("Đảo chiều sau cú sốc — EOD", ls_eod)]):
    v = (1 + ls).cumprod() * VON / 1e6
    fig.add_trace(go.Scatter(x=v.index, y=v, name=ten, line=dict(width=2, color=CHUOI[i])))
fig.add_hline(y=VON / 1e6, line_width=1, line_color="#c3c2b7")
fig.update_layout(title_text=f"{HD} — trong phiên vs EOD, 1 hợp đồng, vốn {VON / 1e6:,.0f} triệu VND<br>"
                  "<sub style='color:#52514e'>Lợi suất theo ngày từ TimeReturn · phí 10.000 VND/HĐ/chiều (giả định) · "
                  "không trượt giá</sub>",
                  yaxis_title="vốn, triệu VND", height=460, hovermode="x unified")
fig

Đọc bảng này với ba lưu ý:

1. **Một năm là một mẫu nhỏ**, và luật phá vỡ khoảng mở cửa được chọn để minh hoạ
   cơ chế, không được tối ưu hay kiểm ngoài mẫu (xem
   [`87`](87_toi_uu_va_walk_forward.ipynb)). Đừng đọc nó như bằng chứng chiến lược
   nào tốt hơn.
2. **Không có trượt giá.** Lệnh thị trường khớp đúng giá mở cửa của thanh 1 phút
   kế tiếp. Với vài trăm vòng mỗi năm, mỗi 0,1 điểm trượt giá mỗi chiều là
   20.000 VND mỗi vòng — cùng cỡ với phí.
3. **Lợi suất tính trên vốn 100 triệu**, trong khi ký quỹ một hợp đồng chưa tới
   một nửa số đó (cột cuối bảng). Cùng một chuỗi lãi/lỗ, vốn nhỏ hơn thì lợi suất phần trăm lớn hơn và rủi
   ro bị gọi ký quỹ cũng lớn hơn.

## Tổng kết

| Bạn cần | Dùng |
|---|---|
| Giá phái sinh theo phiên | `client.eod.derivative.ohlcv("VN30F1M")` — `index_point`, `contract` |
| Dữ liệu 1/5 phút | `client.intraday.derivative.ohlcv(..., interval=)`, cửa sổ `end = start + (max_intraday_days − 1)` |
| Frame → feed | `bt_vn.chuan_bi_frame(..., don_vi_gia="index_point")` sau `pd.concat` |
| Phí, ký quỹ, hệ số nhân | `bt_vn.gan_phi_phai_sinh(cerebro, ...)` — **không** `addcommissioninfo` trần (mục 3) |
| 1 phút → 15 phút | `cerebro.resampledata(..., compression=15, boundoff=1)` |
| Thanh ngày đang hình thành | `cerebro.replaydata(..., timeframe=bt.TimeFrame.Days)` |
| Chuỗi nối không nhảy | điều chỉnh lùi theo `VN30F2M − VN30F1M` ở ngày đáo hạn |

**Mang sang notebook sau:**

1. **Múi giờ**: chỉ một cách bỏ múi giờ là đúng — `tz_convert("Asia/Ho_Chi_Minh")`
   rồi `tz_localize(None)`, như `bt_vn.chuan_bi_frame`. `.values`, đi qua UTC, hay
   đưa thẳng index có múi giờ cho backtrader đều lệch 7 tiếng **mà không báo lỗi**.
2. **Nhãn thanh**: finlens đánh nhãn theo phút bắt đầu; gộp bằng backtrader cần
   `boundoff=1` mới khớp với thanh 5 phút của chính finlens.
3. **Chuỗi nối nhảy ở phiên sau ngày đáo hạn** — đo được bằng `basis`, xử lý bằng
   điều chỉnh cuộn hoặc đứng ngoài quanh ngày đáo hạn.
4. **Phiên dở dang**: EOD không cảnh báo, intraday có. Luôn `bo_phien_chua_dong`.
5. **Kiểm `getvalue()` khi dùng ký quỹ theo tỷ lệ** — lãi/lỗ đúng không có nghĩa
   đường vốn đúng. Gắn phí phái sinh bằng `gan_phi_phai_sinh`: `getvalue()` đúng
   mà `shortcash` vẫn bật thì broker không bao giờ gọi tới nó.

---

**Tiếp theo:** [`89_bang_lenh_cuoi_ngay.ipynb`](89_bang_lenh_cuoi_ngay.ipynb) — chạy
chiến lược đã chọn trên dữ liệu mới nhất và xuất danh sách lệnh cho phiên mai.